# 🏆 V5: The Kaggle Holy Trinity (XGB + Cat + LGBM) 🔥

Notebook ultimate ini memadukan **3 Algoritma Gradient Boosting Terbaik di Dunia** (XGBoost, CatBoost, LightGBM) menggunakan fitur cuaca ekstrem jangka panjang (API).
Kita juga menggunakan trik **Log Transform** pada target TMA untuk menstabilkan prediksi dari fluktuasi banjir ekstrem, serta **100 Trials Optuna** untuk mencari parameter XGBoost paling sakti.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn xgboost catboost lightgbm scikit-learn optuna

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
import optuna
import warnings
import gc

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
sns.set_theme(style="whitegrid")


## 2. Load Data

In [ ]:
# Mount Google Drive (hanya jalan di Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except:
    print("Not running in Google Colab, skipping drive mount.")

path = '/content/drive/MyDrive/ssf-ssds/'
# path = './' # Lokal

print("Loading data...")
train_df = pd.read_csv(path + 'train.csv', parse_dates=['datetime'])
test_df = pd.read_csv(path + 'test.csv')
sample_sub = pd.read_csv(path + 'sample_submission.csv')

lingkungan_df = pd.read_csv(path + 'data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
koordinat_df = pd.read_csv(path + 'data_pendukung/koordinat_pos.csv')


## 3. Ekstraksi Fitur (Turbo Exogenous API) 🌪️
*(Sengaja melewatkan plot EDA di sini agar kita fokus ke pipeline Machine Learning, EDA sudah divisualisasikan di V4)*

Kita membuat akumulasi hujan ekstrem hingga 30 hari!

In [ ]:
print("Memproses Ekstraksi Fitur Lingkungan Ekstrem...")
lingkungan_df = lingkungan_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

def build_exogenous_features(env_df):
    agg_df = env_df[['datetime', 'nama_pos']].copy()
    grouped = env_df.groupby('nama_pos')
    
    # 1. Hujan (Akumulasi Ekstrem)
    if 'rainfall_mm' in env_df.columns:
        agg_df['rain_sum_6h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(6, min_periods=1).sum())
        agg_df['rain_sum_72h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(72, min_periods=1).sum())
        agg_df['rain_sum_168h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(168, min_periods=1).sum())
        agg_df['rain_sum_360h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(360, min_periods=1).sum())
        agg_df['rain_sum_720h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(720, min_periods=1).sum())
        
    # 2. Kelembapan Tanah (Saturasi)
    if 'soil_moisture_0_7cm' in env_df.columns:
        agg_df['soil_mean_6h'] = grouped['soil_moisture_0_7cm'].transform(lambda x: x.rolling(6, min_periods=1).mean())
        agg_df['soil_mean_72h'] = grouped['soil_moisture_0_7cm'].transform(lambda x: x.rolling(72, min_periods=1).mean())
        agg_df['soil_mean_168h'] = grouped['soil_moisture_0_7cm'].transform(lambda x: x.rolling(168, min_periods=1).mean())
        
    # 3. Iklim Global
    for c in ['nino_34', 'mjo_phase']:
        if c in env_df.columns:
            agg_df[f'{c}_mean_720h'] = grouped[c].transform(lambda x: x.rolling(720, min_periods=1).mean())
            
    # 4. Cuaca standar 6 jam
    for c in ['temperature_c', 'wind_speed_kmh', 'surface_pressure_hpa', 'humidity_pct']:
        if c in env_df.columns:
            agg_df[f'{c}_mean_6h'] = grouped[c].transform(lambda x: x.rolling(6, min_periods=1).mean())

    return agg_df

env_feat = build_exogenous_features(lingkungan_df)
env_feat = env_feat[env_feat['datetime'].dt.hour.isin([6, 12, 18])].reset_index(drop=True)

station_elevation = train_df.groupby('nama_pos')['tma_mdpl'].mean().reset_index()
station_elevation.rename(columns={'tma_mdpl': 'elevation_proxy'}, inplace=True)
print("Fitur berhasil dibuat!")


In [ ]:
def prepare_dataset(base_df, env_feat, coords_df, elevation_df):
    df = base_df.copy()
    
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
    
    df = pd.merge(df, env_feat, on=['datetime', 'nama_pos'], how='left')
    df = pd.merge(df, coords_df, on='nama_pos', how='left')
    df = pd.merge(df, elevation_df, on='nama_pos', how='left')
    return df

print("Menyusun Train & Test Set...")
train_full = prepare_dataset(train_df, env_feat, koordinat_df, station_elevation)

test_parsed = test_df.copy()
split_info = test_parsed['id'].str.split(' - ', n=1, expand=True)
test_parsed['datetime'] = pd.to_datetime(split_info[0])
test_parsed['nama_pos'] = split_info[1]
test_full = prepare_dataset(test_parsed, env_feat, koordinat_df, station_elevation)

le = LabelEncoder()
all_stations = pd.concat([train_full['nama_pos'], test_full['nama_pos']]).unique()
le.fit(all_stations)

train_full['nama_pos_encoded'] = le.transform(train_full['nama_pos'])
test_full['nama_pos_encoded'] = le.transform(test_full['nama_pos'])

train_full = train_full.dropna(subset=['tma_mdpl']).reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id']
features = [c for c in train_full.columns if c not in drop_cols]
print(f"Total Fitur: {len(features)}")


## 4. Log Transformation & Setup X, y

In [ ]:
X = train_full[features]
y_asli = train_full['tma_mdpl']

# THE MAGIC TRICK: Mengubah skala Y menjadi Logaritma
# Ini meredam lonjakan anomali sehingga Tree Models (yang sensitif pada outlier) bisa bekerja lebih rileks
y_log = np.log1p(y_asli)

print("Target has been Log-Transformed!")


## 5. Optuna 100-Trials (Untuk XGBoost) 🤖
Kita lepas rantainya hingga 100 trials. Silakan tinggalkan ngopi dulu!

In [ ]:
RUN_OPTUNA = True 

if RUN_OPTUNA:
    def objective(trial):
        param = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'tree_method': 'hist',
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 5, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            'random_state': 42
        }
        
        split_idx = int(len(X) * 0.8)
        X_tr, y_tr = X.iloc[:split_idx], y_log.iloc[:split_idx]
        X_va, y_va = X.iloc[split_idx:], y_log.iloc[split_idx:]
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_va, label=y_va)
        
        model = xgb.train(param, dtrain, num_boost_round=1000, 
                          evals=[(dval, 'val')], early_stopping_rounds=30, verbose_eval=False)
        
        preds = model.predict(dval)
        rmse = root_mean_squared_error(y_va, preds)
        return rmse

    print("Memulai Optuna (100 Trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=100) # LEPAS RANTAI: 100 TRIALS!
    
    print("\nBest parameters found by Optuna:")
    xgb_params = study.best_params
else:
    xgb_params = {'learning_rate': 0.03, 'max_depth': 8, 'subsample': 0.8}

xgb_params.update({'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'random_state': 42})


## 6. Training "The Kaggle Holy Trinity" (K-Fold)
Kita akan melatih XGBoost, LightGBM, dan CatBoost secara paralel!

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store models
xgb_models = []
lgb_models = []
cat_models = []

# Parameter Statis "Anti-Badai" untuk LightGBM
lgb_params = {
    'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.03,
    'num_leaves': 64, 'max_depth': 8, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 1, 'verbose': -1, 'random_state': 42
}

print("Mulai melatih TRINITY K-FOLD (XGB, LGBM, CAT)...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
    print(f"\n========== FOLD {fold+1} ==========")
    X_tr, y_tr = X.iloc[train_idx], y_log.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y_log.iloc[val_idx]
    
    # 1. XGBoost
    dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr)
    dval_xgb = xgb.DMatrix(X_va, label=y_va)
    model_xgb = xgb.train(xgb_params, dtrain_xgb, num_boost_round=3000, 
                          evals=[(dtrain_xgb, 'train'), (dval_xgb, 'val')],
                          early_stopping_rounds=100, verbose_eval=500)
    xgb_models.append(model_xgb)
    
    # 2. LightGBM
    dtrain_lgb = lgb.Dataset(X_tr, label=y_tr)
    dval_lgb = lgb.Dataset(X_va, label=y_va, reference=dtrain_lgb)
    model_lgb = lgb.train(lgb_params, dtrain_lgb, num_boost_round=3000,
                          valid_sets=[dtrain_lgb, dval_lgb],
                          callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=500)])
    lgb_models.append(model_lgb)
    
    # 3. CatBoost
    model_cat = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=8,
                                  eval_metric='RMSE', random_seed=42, verbose=500)
    # CatBoost secara native mengenali kolom string/kategorikal, 
    # karena kita sudah label encode, kita beritahu indeks fitur kategorikalnya
    cat_features_idx = [list(X_tr.columns).index('nama_pos_encoded')]
    model_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), 
                  cat_features=cat_features_idx, early_stopping_rounds=100)
    cat_models.append(model_cat)


## 7. Ensemble Inference (Blending) ⚔️
Setiap algoritma memiliki kelemahannya sendiri. Namun jika kita gabungkan tebakan ketiganya (Blending), kelemahan itu akan saling menutupi!

In [ ]:
X_test = test_full[features]
dtest_xgb = xgb.DMatrix(X_test)

preds_xgb = np.zeros(len(X_test))
preds_lgb = np.zeros(len(X_test))
preds_cat = np.zeros(len(X_test))

# Prediksi menggunakan ensemble tiap Fold
for i in range(5):
    preds_xgb += xgb_models[i].predict(dtest_xgb) / 5
    preds_lgb += lgb_models[i].predict(X_test) / 5
    preds_cat += cat_models[i].predict(X_test) / 5

# KEMBALIKAN NILAI DARI LOG KE SKALA ASLI TMA MENGGUNAKAN EXPM1
preds_xgb_asli = np.expm1(preds_xgb)
preds_lgb_asli = np.expm1(preds_lgb)
preds_cat_asli = np.expm1(preds_cat)

# BLENDING THE TRINITY: 40% XGBoost, 30% LightGBM, 30% CatBoost
final_preds = (0.4 * preds_xgb_asli) + (0.3 * preds_lgb_asli) + (0.3 * preds_cat_asli)

test_full['tma_mdpl'] = final_preds

# Format Submission
sub_pred = test_full[['id', 'tma_mdpl']].copy()
sub = pd.merge(sample_sub[['id']], sub_pred, on='id', how='left')
sub['tma_mdpl'] = sub['tma_mdpl'].fillna(method='ffill').fillna(method='bfill')

display(sub.head(10))
sub.to_csv('submission_v5_holy_trinity.csv', index=False)
print("\n✅ THE HOLY TRINITY COMPLETED! File 'submission_v5_holy_trinity.csv' siap membawa Anda menembus < 1.60!")
